# 01 - Setup

Run once on the remote H200 Jupyter server. Installs dependencies, downloads the model and dataset, verifies GPU access, and runs a smoke test with nnsight.

**Fully resumable** - safe to re-run all cells if the session disconnects.

In [16]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/10-4-2026")
REPO_DIR = WORKSPACE / "legibility"

# Clone or pull shared utilities
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/JackHopkins/legibility.git", str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

# Create data directories
for d in [CACHE_DIR, COT_CACHE, INTERVENTION_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Directories created:")
for d in [CACHE_DIR, COT_CACHE, INTERVENTION_CACHE, RESULTS_DIR, FIGURES_DIR]:
    print(f"  {d}")

Updating 6f4c497..3ee9cdf
Fast-forward
 01_setup.ipynb      | 79 ++++++++++++++++++++++++++++-------------------------
 lib/intervention.py |  3 +-
 2 files changed, 44 insertions(+), 38 deletions(-)
Directories created:
  /workspace/10-4-2026/cache
  /workspace/10-4-2026/cache/cots
  /workspace/10-4-2026/cache/interventions
  /workspace/10-4-2026/results
  /workspace/10-4-2026/figures


From https://github.com/JackHopkins/legibility
   6f4c497..3ee9cdf  main       -> origin/main


In [17]:
# Install dependencies
!pip install -q nnsight transformers datasets tqdm matplotlib seaborn

In [18]:
# Verify GPU
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Interventions require GPU.")

CUDA available: True
Device: NVIDIA H200
Memory: 150.0 GB


In [19]:
# Download and cache GSM8K
from datasets import load_dataset
from lib.config import DATASET_NAME, DATASET_CONFIG, DATASET_SPLIT

ds = load_dataset(DATASET_NAME, DATASET_CONFIG, split=DATASET_SPLIT)
print(f"GSM8K loaded: {len(ds)} examples")
print(f"Example: {ds[0]['question'][:100]}...")
print(f"Answer: {ds[0]['answer'][-30:]}")

GSM8K loaded: 7473 examples
Example: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How m...
Answer: ther in April and May.
#### 72


In [20]:
# Download and cache the primary model with nnsight
print(f"Loading {MODEL_NAME} with nnsight...")

try:
    from nnsight import LanguageModel
    model = LanguageModel(MODEL_NAME, device_map="auto", dispatch=True, dtype=torch.bfloat16)
    tokenizer = model.tokenizer
    NNSIGHT_WORKS = True
    print(f"nnsight loaded {MODEL_NAME} successfully")
except Exception as e:
    print(f"nnsight failed: {e}")
    print("Falling back to raw transformers...")
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, device_map="auto", torch_dtype=torch.bfloat16
    )
    model.eval()
    NNSIGHT_WORKS = False
    print(f"Loaded {MODEL_NAME} with transformers")

Loading Qwen/Qwen3-4B with nnsight...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

nnsight loaded Qwen/Qwen3-4B successfully


In [21]:
# Inspect model architecture
if NNSIGHT_WORKS:
    inner = model.model
else:
    inner = model.model

print(f"Number of layers: {len(inner.layers)}")
print(f"Embed tokens shape check: {inner.embed_tokens.weight.shape}")
print(f"\nFirst layer type: {type(inner.layers[0])}")
print(f"\nFull model structure (top-level):")
for name, _ in inner.named_children():
    print(f"  model.{name}")

Number of layers: 36
Embed tokens shape check: torch.Size([151936, 2560])

First layer type: <class 'nnsight.intervention.envoy.Envoy'>

Full model structure (top-level):
  model.embed_tokens
  model.layers
  model.norm
  model.rotary_emb


In [22]:
# Smoke test: probe layer input structure and verify shapes
test_text = "<|im_start|>user\nWhat is 2+2?<|im_end|>\n<|im_start|>assistant\nThe answer is:"
input_ids = tokenizer(test_text, return_tensors="pt").input_ids
print(f"Input shape: {input_ids.shape}")

if NNSIGHT_WORKS:
    input_ids_device = input_ids.to(model.device)

    # First: probe the layer input structure
    with model.trace(input_ids_device):
        emb = model.model.embed_tokens.output.save()
        layer0_in_full = model.model.layers[0].input.save()
        logits = model.lm_head.output.save()

    print(f"Embedding output shape: {emb.shape}")
    print(f"Logits shape: {logits.shape}")
    print(f"\nLayer 0 input type: {type(layer0_in_full)}")
    if isinstance(layer0_in_full, (tuple, list)):
        print(f"Layer 0 input length: {len(layer0_in_full)}")
        for i, item in enumerate(layer0_in_full):
            if hasattr(item, 'shape'):
                print(f"  [{i}] shape: {item.shape}")
            elif isinstance(item, (tuple, list)):
                print(f"  [{i}] tuple/list len={len(item)}")
                for j, sub in enumerate(item):
                    if hasattr(sub, 'shape'):
                        print(f"    [{i}][{j}] shape: {sub.shape}")
                    else:
                        print(f"    [{i}][{j}] type: {type(sub)}")
            else:
                print(f"  [{i}] type: {type(item)}")
    else:
        print(f"  shape: {layer0_in_full.shape}")

    # Now test that we can access hidden states at [batch, seq, hidden]
    # layers[k].input[0] should be the hidden_states tensor
    with model.trace(input_ids_device):
        layer0_hidden = model.model.layers[0].input[0].save()

    print(f"\nLayer 0 input[0] shape: {layer0_hidden.shape}")
    print(f"Expected: [1, {input_ids.shape[1]}, {HIDDEN_SIZE}]")

    # Check embedding vs layer 0 input
    diff = (emb - layer0_hidden).abs().max().item()
    print(f"\nMax diff (embed vs layer0 input[0]): {diff:.6f}")
    if diff < 1e-3:
        print("Embedding feeds directly into layer 0 -- intervention is clean.")
    else:
        print("NOTE: There is a transform between embedding and layer 0 (likely RMSNorm).")
        print("This is expected for some architectures. The intervention still replaces")
        print("the residual with the embedding-only representation.")
else:
    device = next(model.parameters()).device
    input_ids_device = input_ids.to(device)

    captured = {}

    def capture_embed_hook(module, input, output):
        captured["embedding"] = output.detach().clone()

    def capture_layer0_hook(module, args):
        captured["layer0_input"] = args[0].detach().clone()

    h1 = model.model.embed_tokens.register_forward_hook(capture_embed_hook)
    h2 = model.model.layers[0].register_forward_pre_hook(capture_layer0_hook)

    with torch.no_grad():
        outputs = model(input_ids_device)

    h1.remove()
    h2.remove()

    print(f"Embedding output shape: {captured['embedding'].shape}")
    print(f"Layer 0 input shape: {captured['layer0_input'].shape}")
    print(f"Logits shape: {outputs.logits.shape}")
    print(f"\nExpected: [1, {input_ids.shape[1]}, {HIDDEN_SIZE}]")

    diff = (captured["embedding"] - captured["layer0_input"]).abs().max().item()
    print(f"\nMax diff (embed vs layer0 input): {diff:.6f}")

print("\nSmoke test complete.")

Input shape: torch.Size([1, 19])
Embedding output shape: torch.Size([1, 19, 2560])
Logits shape: torch.Size([1, 19, 151936])

Layer 0 input type: <class 'torch.Tensor'>
  shape: torch.Size([1, 19, 2560])

Layer 0 input[0] shape: torch.Size([19, 2560])
Expected: [1, 19, 2560]

Max diff (embed vs layer0 input[0]): 0.000000
Embedding feeds directly into layer 0 -- intervention is clean.

Smoke test complete.


In [ ]:
# Smoke test: verify intervention works (zeroed_layer_0)
if NNSIGHT_WORKS:
    input_ids_device = input_ids.to(model.device)

    # Normal forward pass
    with model.trace(input_ids_device):
        normal_logits = model.lm_head.output[0, -1, :].save()

    # Zeroed at layer 0: replace last-position residual with raw embedding
    with model.trace(input_ids_device):
        embedding = model.model.embed_tokens.output[0, -1, :].clone().save()
        # .input is hidden_states directly [batch, seq, hidden]
        model.model.layers[0].input[0, -1, :] = embedding
        zeroed_logits = model.lm_head.output[0, -1, :].save()

    normal_probs = torch.softmax(normal_logits.float(), dim=-1)
    zeroed_probs = torch.softmax(zeroed_logits.float(), dim=-1)

    # KL divergence
    kl = torch.sum(normal_probs * (torch.log(normal_probs + 1e-10) - torch.log(zeroed_probs + 1e-10))).item()
    print(f"KL(normal || zeroed_layer_0) = {kl:.4f}")

    # Top predictions
    for label, probs in [("Normal", normal_probs), ("Zeroed L0", zeroed_probs)]:
        top5_probs, top5_ids = torch.topk(probs, 5)
        tokens = [tokenizer.decode([tid]).strip() for tid in top5_ids.tolist()]
        print(f"\n{label} top-5: {list(zip(tokens, [f'{p:.4f}' for p in top5_probs.tolist()]))}")

    print("\nIntervention smoke test complete. nnsight is working.")
else:
    print("Skipping nnsight intervention test (using hook fallback).")
    print("Testing hook-based intervention...")

    from lib.intervention import load_model, forward_pass_logits
    normal_logits = forward_pass_logits(test_text)
    zeroed_logits = forward_pass_logits(test_text, zero_at_layer=0)

    normal_probs = torch.softmax(normal_logits, dim=-1)
    zeroed_probs = torch.softmax(zeroed_logits, dim=-1)

    kl = torch.sum(normal_probs * (torch.log(normal_probs + 1e-10) - torch.log(zeroed_probs + 1e-10))).item()
    print(f"KL(normal || zeroed_layer_0) = {kl:.4f}")
    print("Hook-based intervention smoke test complete.")

In [ ]:
print("Setup complete. Ready to proceed with 02_generate_cots.ipynb.")